# Basic workflow with ManGO

## Authentication with iCommands

<div class="alert alert-block alert-info">
<h3>Every seven days</h3>
    
1. Go to https://mango.kuleuven.be/
2. In the tab of your zone, click on "How to connect"
3. Copy the snippet using iron".

<font size=3>Then **paste the snippet** in the cell below, right under `%%bash`, like the (anonimized) example below.</font>

**You don't need to do this every time: The authentication lasts 7 days.**
</div>

In [1]:
%%bash
iron auth u0155029 ghum ghum.irods.icts.kuleuven.be

You are authenticated using a refresh token.


## Setting up

Set up ManGO and load any other libraries you need.

In [2]:
import os
import ssl
from irods.session import iRODSSession
try:
    env_file = os.environ['IRODS_ENVIRONMENT_FILE']
except KeyError:
    env_file = os.path.expanduser('~/.irods/irods_environment.json')


Since we are working interactively we will create an `irods.session.iRODSSession` object and then close it at the end of the notebook with `session.cleanup()`. If you were working on a script, you could run all your code inside a `with` statement.

In [3]:
session = iRODSSession(irods_env_file=env_file)

The final step to set up your environment is to define your working directory in a variable. For this notebook, it's "/ghum/home/ghum_pilot034/", but for your own project it might be **different**.

In [3]:
home_dir = "/ghum/home/KULeuvenLibraries_Collections_As_Data"

In [4]:
dataset_dir = home_dir + "DATASET_9" # it might be a different name
output_dir = home_dir + "TEAM_6" # it might be a different name

## WRITE YOUR CODE BELOW THIS

In [4]:
scratch = "/scratch/leuven/387/vsc38772/IE63547717"

In [5]:
contents = os.listdir(scratch)
print(contents)

['manifest.json', 'REP63547718', 'OCR']


In [6]:
import cv2
import shutil
import pandas as pd
from pathlib import Path

# 1. Define paths and load the Excel file
base_path = Path("/scratch/leuven/387/vsc38772")
target_dirs = [
    "IE63547717/REP63547718",
    "IE63529066/REP63529067",
    "IE63532297/REP63532298"
    
]


output_base_dir = Path("./extracted_illustrations_final")

excel_path = Path("./Thomson-reconcile_v3.xlsx") 

df = pd.read_excel(excel_path)

valid_filenames = set(df['image_file_name_full'].astype(str).str.strip())

# 2. Loop through each target directory
for target in target_dirs:
    input_dir = base_path / target
    
    # Create output subdirectory
    folder_name = target.split('/')[0]
    output_dir = output_base_dir / folder_name
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Get all JPG files
    all_images = list(input_dir.glob("*.jpg"))
    
    # FILTER: Only keep the images whose names appear in fixed valid_filenames list
    images_to_process = [img for img in all_images if img.name in valid_filenames]
    
    print(f"Directory {target}: Found {len(images_to_process)} matches in Excel (out of {len(all_images)} total). Processing...")
    
    # 3. Process only the filtered images
    for img_path in images_to_process:
        image = cv2.imread(str(img_path))
        
        if image is None:
            continue
            
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, 50, 150)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
        dilated = cv2.dilate(edges, kernel, iterations=2)
        contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        min_area = 25000 
        largest_contour = None
        max_area = 0
        
        for contour in contours:
            x, y, w, h = cv2.boundingRect(contour)
            area = w * h
            if area > min_area and area > max_area:
                max_area = area
                largest_contour = (x, y, w, h)
                
        if largest_contour:
            x, y, w, h = largest_contour
            cropped = image[y:y+h, x:x+w]
            
            # Save keeping the original filename
            output_path = output_dir / f"{img_path.stem}_i{img_path.suffix}"
            cv2.imwrite(str(output_path), cropped)

# 4. Zip the entire output base folder for easy downloading
shutil.make_archive("extracted_illustrations_final", "zip", output_base_dir)
print("\nProcessing complete! You can now download 'extracted_illustrations_final.zip'.")


Directory IE63547717/REP63547718: Found 145 matches in Excel (out of 802 total). Processing...
Directory IE63529066/REP63529067: Found 130 matches in Excel (out of 804 total). Processing...
Directory IE63532297/REP63532298: Found 140 matches in Excel (out of 690 total). Processing...

Processing complete! You can now download 'extracted_illustrations_final.zip'.


In [7]:
filtered_df = df[df['text_id'] == 'IE63529066']

distinct_filenames = filtered_df['image_file_name_full'].unique().tolist()

print(f"Found {len(distinct_filenames)} distinct images for text_id 63547717:\n")
for name in distinct_filenames:
    print(name)

Found 130 distinct images for text_id 63547717:

FL63529073_KULGB033483_00000006.jpg
FL63529117_KULGB033483_00000050.jpg
FL63529123_KULGB033483_00000056.jpg
FL63529130_KULGB033483_00000063.jpg
FL63529138_KULGB033483_00000071.jpg
FL63529146_KULGB033483_00000079.jpg
FL63529150_KULGB033483_00000083.jpg
FL63529155_KULGB033483_00000088.jpg
FL63529157_KULGB033483_00000090.jpg
FL63529159_KULGB033483_00000092.jpg
FL63529161_KULGB033483_00000094.jpg
FL63529165_KULGB033483_00000098.jpg
FL63529170_KULGB033483_00000103.jpg
FL63529174_KULGB033483_00000107.jpg
FL63529178_KULGB033483_00000111.jpg
FL63529190_KULGB033483_00000123.jpg
FL63529193_KULGB033483_00000126.jpg
FL63529196_KULGB033483_00000129.jpg
FL63529208_KULGB033483_00000141.jpg
FL63529218_KULGB033483_00000151.jpg
FL63529224_KULGB033483_00000157.jpg
FL63529226_KULGB033483_00000159.jpg
FL63529233_KULGB033483_00000166.jpg
FL63529235_KULGB033483_00000168.jpg
FL63529242_KULGB033483_00000175.jpg
FL63529244_KULGB033483_00000177.jpg
FL63529258_KULG

## CLEAN UP 
<div class="alert alert-block alert-warning">
    <font size=4><b>Do not forget to clean up your session!</b></font>
</div>

In [8]:
# leave this cell at the end and running every time you are done
session.cleanup()